### base approach

In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer

from xgboost import XGBClassifier

### Load and basic cleaning

In [2]:

df = pd.read_csv("datasets/dataset1_text_rich_transactions_harder_v3_prophet_ready.csv")



In [3]:
print("Rows after loading:", len(df))

print("After dropna(description, category_label, transaction_date):",
      df.dropna(subset=["description", "category_label", "transaction_date"]).shape[0])

print("Unique currencies:", df["currency"].astype(str).str.upper().value_counts())

tmp = df.copy()
tmp["transaction_date"] = pd.to_datetime(tmp["transaction_date"],
                                         format="%d-%m-%Y",
                                         errors="coerce")
print("Non-null dates after parse:", tmp["transaction_date"].notna().sum())

tmp["amount"] = pd.to_numeric(tmp["amount"], errors="coerce")
print("Non-null amounts:", tmp["amount"].notna().sum())

print("Final df shape before X/y:", df.shape)


Rows after loading: 10000
After dropna(description, category_label, transaction_date): 9750
Unique currencies: currency
INR    10000
Name: count, dtype: int64
Non-null dates after parse: 0
Non-null amounts: 10000
Final df shape before X/y: (10000, 13)


In [4]:
# Parse dates (dd-mm-YYYY)
df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors="coerce")

# Drop rows with critical missing values
df = df.dropna(subset=["description", "category_label", "transaction_date"])

# keep only INR
if "currency" in df.columns:
    df = df[df["currency"].astype(str).str.upper() == "INR"].copy()

# Fill missing vendor with placeholder
if "vendor_name" in df.columns:
    df["vendor_name"] = df["vendor_name"].fillna("UNKNOWNVENDOR")

In [5]:
print("Rows after loading:", len(df))

print("After dropna(description, category_label, transaction_date):",
      df.dropna(subset=["description", "category_label", "transaction_date"]).shape[0])

print("Unique currencies:", df["currency"].astype(str).str.upper().value_counts())

tmp = df.copy()
tmp["transaction_date"] = pd.to_datetime(tmp["transaction_date"],
                                         format="%d-%m-%Y",
                                         errors="coerce")
print("Non-null dates after parse:", tmp["transaction_date"].notna().sum())

tmp["amount"] = pd.to_numeric(tmp["amount"], errors="coerce")
print("Non-null amounts:", tmp["amount"].notna().sum())

print("Final df shape before X/y:", df.shape)


Rows after loading: 9750
After dropna(description, category_label, transaction_date): 9750
Unique currencies: currency
INR    9750
Name: count, dtype: int64
Non-null dates after parse: 9750
Non-null amounts: 9750
Final df shape before X/y: (9750, 13)


In [6]:
def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.lower()
    # Noise tokens similar to your POC notebooks
    noise_tokens = [
        r"\bfy24\b", r"\bfy25\b", r"\bq1\b", r"\bq2\b",
        r"inv\d*", r"rcpt\d*", r"taxinv\d*", r"\bref\s*\d+",
        r"\bsubs\b", r"\badv\b", r"\bimps\b", r"\bupi\b",
        r"\btxnid\b", r"\btxn\b", r"\brcpt\b"
    ]
    for pat in noise_tokens:
        s = re.sub(pat, " ", s)
    # Keep alphanumerics and spaces
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def clean_vendor(v: str) -> str:
    if not isinstance(v, str):
        return ""
    v = v.upper()
    # Remove punctuation and common suffixes
    v = re.sub(r"[^A-Z0-9]+", " ", v)
    v = re.sub(r"\bPVT\b|\bPRIVATE\b|\bLTD\b|\bLIMITED\b|\bINDIA\b", " ", v)
    v = re.sub(r"\s+", " ", v).strip()
    return v

df["description_clean"] = df["description"].apply(clean_text)
df["vendor_clean"] = df["vendor_name"].apply(clean_vendor)

### Numeric & date features

In [7]:
df["month"] = df["transaction_date"].dt.month
df["dow"] = df["transaction_date"].dt.dayofweek  # 0=Monday

# Amount (ensure numeric)
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
df = df.dropna(subset=["amount"])

### Combine text fields

In [8]:
df["textcombo"] = (
    df["description_clean"].fillna("") + " " +
    df["vendor_clean"].fillna("")
)

TEXTCOL = "textcombo"
NUMCOLS = ["amount", "month", "dow"]
TARGETCOL = "category_label"

X = df[[TEXTCOL] + NUMCOLS].copy()
y = df[TARGETCOL].copy()

### Label encoding & split

In [9]:
le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc,
    test_size=0.2,
    stratify=y_enc,
    random_state=42
)

### Preprocess + XGBoost model

In [10]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=3
)

preprocess = ColumnTransformer(
    transformers=[
        ("text", tfidf, TEXTCOL),
        ("num", "passthrough", NUMCOLS),
    ],
    remainder="drop"
)

xgb_clf = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    max_depth=8,
    learning_rate=0.1,
    n_estimators=300,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1
)

pipe = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("clf", xgb_clf),
    ]
)

### Train and evaluate

In [11]:
pipe.fit(X_train, y_train)

y_pred_enc = pipe.predict(X_test)
y_test_str = le.inverse_transform(y_test)
y_pred_str = le.inverse_transform(y_pred_enc)

print("TF-IDF + XGBoost (Type 1)")
print(classification_report(y_test_str, y_pred_str))
print("Macro F1:",
      f1_score(y_test_str, y_pred_str, average="macro"))

TF-IDF + XGBoost (Type 1)
                 precision    recall  f1-score   support

Exempt Services       1.00      1.00      1.00        53
    IT Services       0.98      0.96      0.97       417
          Meals       0.92      0.99      0.96       110
Office Supplies       1.00      1.00      1.00       256
           Rent       1.00      1.00      1.00       302
       Software       1.00      1.00      1.00       229
       Training       0.95      0.92      0.94       152
         Travel       1.00      0.99      1.00       181
      Utilities       0.96      0.98      0.97       250

       accuracy                           0.98      1950
      macro avg       0.98      0.98      0.98      1950
   weighted avg       0.98      0.98      0.98      1950

Macro F1: 0.9814600098451997


In [12]:

def predict_category(description: str, vendorname: str,
                     amount: float, transactiondate: str):
    dt = pd.to_datetime(transactiondate, format="%d-%m-%Y", errors="coerce")
    month = dt.month
    dow = dt.dayofweek

    desc_clean = clean_text(description)
    vend_clean = clean_vendor(vendorname)
    textcombo = desc_clean + " " + vend_clean

    x_row = pd.DataFrame([{
        TEXTCOL: textcombo,
        "amount": amount,
        "month": month,
        "dow": dow
    }])

    pred_enc = pipe.predict(x_row)[0]
    return le.inverse_transform([pred_enc])[0]
